In [ ]:
# FAST-UAV - Supply Chain Integrated Design of Experiments
*Coupled Analysis of Drone Sizing and Supply Chain Metrics*

이 노트북은 물리적 드론 사이징(Payload, Endurance 등)과 공급망 분석(Cost, Lead Time, Risk)을 통합하여 실험계획법(DOE)을 수행합니다. 
`fastoad`를 사용한 사이징 최적화 결과에 대해 `fastuav.models.supply_chain` 패키지를 사용하여 실시간으로 BOM 및 공급망 메트릭을 계산합니다.

In [1]:
# Import Required Libraries
import os
import os.path as pth
import pandas as pd
import numpy as np
import shutil
import xml.etree.ElementTree as ET
import plotly.express as px
import plotly.graph_objects as go
import fastoad.api as oad

# Import new Supply Chain Logic
try:
    from fastuav.models.supply_chain.model import run_supply_chain_scenario
except ImportError:
    # Adding path for local development
    import sys
    current_dir = os.getcwd()
    src_path = pth.abspath(pth.join(current_dir, '..', '..'))
    if src_path not in sys.path:
        sys.path.append(src_path)
    from fastuav.models.supply_chain.model import run_supply_chain_scenario

from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

c:\Users\user\.conda\envs\FastUAV_edit\lib\site-packages\stdatm\__init__.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
# --- Path Setup ---
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = "./workdir"
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "multirotor_mdo.yaml")
SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_quadcopter.xml")

if not os.path.exists(WORK_FOLDER_PATH):
    os.makedirs(WORK_FOLDER_PATH)

print("Configuration:", CONFIGURATION_FILE)
print("Source input:", SOURCE_FILE)

# Copy source samples for Supply Chain
catalog_csv = pth.join(SOURCE_FOLDER_PATH, 'supplier_parts_catalog_sample.csv')
map_csv = pth.join(SOURCE_FOLDER_PATH, 'bom_to_part_family_map_sample.csv')
assump_csv = pth.join(SOURCE_FOLDER_PATH, 'cost_leadtime_assumptions_sample.csv')

# Load Baseline SC Data
catalog_df_base = pd.read_csv(catalog_csv)
map_df_base = pd.read_csv(map_csv)
assump_df_base = pd.read_csv(assump_csv)
assump_base = dict(zip(assump_df_base['key'], assump_df_base['value']))

Configuration: ./data\configurations\multirotor_mdo.yaml
Source input: ./data\source_files\problem_inputs_quadcopter.xml


In [3]:
def generate_bom_from_output(output_xml_path):
    """
    Generate DataFrame BOM from FAST-OAD output XML.
    (This logic mirrors Section 5 of 1d_Quadcopter_Design_Continuous.ipynb)
    """
    if not os.path.exists(output_xml_path):
        raise FileNotFoundError(f'{output_xml_path} not found')
        
    root = ET.parse(output_xml_path).getroot()

    def get_float(path, default=None):
        node = root.find(path)
        if node is None: return default
        try: return float(node.text)
        except: return default

    def get_float_any(paths, default=None):
        for p in paths:
            v = get_float(p, None)
            if v is not None: return v
        return default

    # 1. Quantities
    n_prop = get_float_any(['./data/propulsion/multirotor/propeller/number', './data/geometry/multirotor/arms/number'], 4.0)
    n_arm = get_float_any(['./data/geometry/multirotor/arms/number'], n_prop)
    n_motor = n_prop
    n_esc = n_prop

    # 2. Masses (kg)
    m_battery = get_float_any(['./data/weight/propulsion/multirotor/battery/mass'], 0.0)
    m_esc = get_float_any(['./data/weight/propulsion/multirotor/esc/mass'], 0.0)
    m_motor = get_float_any(['./data/weight/propulsion/multirotor/motor/mass'], 0.0)
    m_prop = get_float_any(['./data/weight/propulsion/multirotor/propeller/mass'], 0.0)
    
    # 3. Perf References
    battery_energy_kj = get_float_any(['./data/propulsion/multirotor/battery/energy', './data/propulsion/multirotor/battery/energy/estimated'], None)
    esc_max_power_w = get_float_any(['./data/propulsion/multirotor/esc/max', './data/propulsion/multirotor/esc/power/max'], None)
    motor_max_torque_nm = get_float_any(['./data/propulsion/multirotor/motor/torque/max', './data/propulsion/multirotor/motor/torque/max/estimated'], None)
    motor_length_m = get_float_any(['./data/propulsion/multirotor/motor/length', './data/propulsion/multirotor/motor/length/estimated'], None)
    prop_diameter_m = get_float_any(['./data/propulsion/multirotor/propeller/diameter', './data/propulsion/multirotor/propeller/diameter/estimated'], None)
    arm_length_m = get_float_any(['./data/geometry/multirotor/arms/length', './data/geometry/arms/length'], None)

    rows = [
        {
            'component_id': 'battery_pack', 'part_family': 'battery', 'quantity': 1,
            'unit_mass_kg': m_battery, 'total_mass_kg': m_battery,
            'required_perf_key': 'energy_kJ', 'required_perf_min': battery_energy_kj,
        },
        {
            'component_id': 'motor', 'part_family': 'motor', 'quantity': int(n_motor),
            'unit_mass_kg': m_motor, 'total_mass_kg': m_motor * n_motor,
            'required_perf_key': 'max_torque_Nm', 'required_perf_min': motor_max_torque_nm,
        },
        {
            'component_id': 'esc', 'part_family': 'esc', 'quantity': int(n_esc),
            'unit_mass_kg': m_esc, 'total_mass_kg': m_esc * n_esc,
            'required_perf_key': 'max_power_W', 'required_perf_min': esc_max_power_w,
        },
        {
            'component_id': 'propeller', 'part_family': 'propeller', 'quantity': int(n_prop),
            'unit_mass_kg': m_prop, 'total_mass_kg': m_prop * n_prop,
            'required_perf_key': 'diameter_m', 'required_perf_min': prop_diameter_m,
        },
        {
            'component_id': 'frame_arms', 'part_family': 'frame_arms', 'quantity': int(n_arm),
            'unit_mass_kg': 0.0, 'total_mass_kg': 0.0, # Frame mass logic is complex in fastoad, keep simplified
            'required_perf_key': 'arm_length_m', 'required_perf_min': arm_length_m,
        },
    ]
    return pd.DataFrame(rows)

In [4]:
def evaluate_point(
    # --- Physical Params ---
    payload_mass_kg: float,
    hover_duration_min: float,
    # --- Supply Chain Params ---
    risk_tolerance: float,
    logistics_buffer: int,
):
    """
    Evaluates a single design point.
    1. Updates Physical inputs -> Runs Optimization.
    2. Generates BOM from optimized vehicle.
    3. Runs Supply Chain Scenario.
    4. Returns consolidated metrics.
    """
    # 1. Update Input XML
    temp_input_xml = pth.join(WORK_FOLDER_PATH, "temp_inputs_doe.xml")
    
    # Read base XML
    oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True) # Reset base
    base_input = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml") 
    
    # Use fastoad API to update values if possible, or string replace
    # Here we manipulate the XML via fastoad variable viewer logic or directly via text/xml api
    # Creating a small override inputs file is cleaner if fastoad supports it, but here we just overwrite problem_inputs.xml
    
    # Let's use simple text replacement or XML tree for speed/reliability on the known file structure
    tree = ET.parse(base_input)
    root = tree.getroot()
    
    # Helper to find and update
    def update_val(path_suffix, new_val):
        # Scan all variables for matching id/name
        for var in root.findall(".//variable"):
            name = var.find("name").text
            if name.endswith(path_suffix):
                var.find("value").text = str(new_val)
                return True
        return False
        
    # Update Payload
    # "mission:operational:main_route:payload:mass"
    # Note: In problem_inputs.xml, names are like "mission:operational:main_route:payload:mass"
    # Find exact matches
    found_puck = update_val("mission:operational:main_route:payload:mass", payload_mass_kg)
    # Update Hover Duration
    # "mission:sizing:main_route:hover:duration" (min)
    found_hover = update_val("mission:sizing:main_route:hover:duration", hover_duration_min)
    
    tree.write(temp_input_xml)
    
    # 2. Run Optimization
    # We need to tell optimize_problem to use our temp input.
    # The configuration file points to a specific input file. We can override it or modify the yaml.
    # Easier: Just start from a fresh yaml that points to our temp input?
    # Or, overwrite problem_inputs.xml which is what the default yaml often uses (if generated).
    # Actually, multirotor_mdo.yaml usually points to problem_inputs.xml in the workdir 
    # IF we ran generate_inputs. Let's check.
    # The standard flow is:
    #   oad.generate_inputs(CONF, SOURCE) -> writes to workdir/problem_inputs.xml
    #   oad.optimize_problem(CONF) -> reads workdir/problem_inputs.xml
    
    # So we just overwrote problem_inputs.xml (by saving tree to it).
    tree.write(base_input)
    
    problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)
    # The problem runs and writes output to workdir/problem_outputs.xml
    
    output_xml = pth.join(WORK_FOLDER_PATH, "problem_outputs.xml")
    if not os.path.exists(output_xml):
        return None # Failed
        
    # Get MTOW for reference
    out_tree = ET.parse(output_xml)
    mtow = float(out_tree.getroot().find(".//variable[name='data:weight:mtow']/value").text)
    
    # 3. Generate BOM
    bom_df = generate_bom_from_output(output_xml)
    
    # 4. Run Supply Chain
    # Map risk_tolerance (0.0-1.0) to risk_cost_multiplier (e.g. 0.0 -> high multiplier? No.
    # Risk Tolerance usually means "I accept risk". 
    # Let's treat the input as "Risk Aversion Factor" (Multiplier). 
    # Input is "risk_tolerance" -> let's map it: 
    # High tolerance = Low Multiplier (don't pay for risk). Low tolerance = High Multiplier.
    # Let's just pass "risk_cost_multiplier" directly as the DOE var.
    
    selection_df, summary_df, mat_df = run_supply_chain_scenario(
        bom_df=bom_df,
        catalog_df=catalog_df_base,
        map_df=map_df_base,
        quality_buffer_ratio=0.05,
        scrap_ratio=0.03,
        risk_cost_multiplier=risk_tolerance, 
        logistics_buffer_days=logistics_buffer,
        performance_margin_factor=1.0,
        selection_strategy='min_cost',
        use_continuous_model=True,
        component_overrides={},
        critical_path_mode='max'
    )
    
    total_cost = summary_df[summary_df['metric']=='total_cost_usd']['value'].values[0]
    lead_time = summary_df[summary_df['metric']=='total_lead_time_days']['value'].values[0]
    total_risk_val = summary_df[summary_df['metric']=='total_cost_risk_adjusted_usd']['value'].values[0]
    
    return {
        "payload_kg": payload_mass_kg,
        "hover_min": hover_duration_min,
        "risk_mult": risk_tolerance,
        "logistics_days": logistics_buffer,
        "mtow_kg": mtow,
        "total_cost_usd": total_cost,
        "lead_time_days": lead_time,
        "risk_adj_cost_usd": total_risk_val
    }

In [5]:
import numpy as np
import pandas as pd
import tqdm

# Define DOE Ranges
payloads = np.linspace(0.5, 3.0, 4)  # kg
hovers = np.linspace(10, 30, 3)      # min
risk_multipliers = [1.0, 1.2, 1.5]
logistics_buffers = [0, 5, 10]       # days

doe_results = []

# Simple Full Factorial
total_runs = len(payloads) * len(hovers) * len(risk_multipliers) * len(logistics_buffers)

print(f"Starting DOE with {total_runs} evaluations...")

with tqdm.tqdm(total=total_runs) as pbar:
    for p in payloads:
        for h in hovers:
            for r in risk_multipliers:
                for l in logistics_buffers:
                    try:
                        res = evaluate_point(
                            payload_mass_kg=p,
                            hover_duration_min=h,
                            risk_tolerance=r,
                            logistics_buffer=l
                        )
                        if res:
                            doe_results.append(res)
                    except Exception as e:
                        print(f"Failed run: P={p}, H={h} -> {e}")
                    pbar.update(1)

# Convert to DataFrame
df_results = pd.DataFrame(doe_results)
df_results.head()

Starting DOE with 108 evaluations...


  0%|          | 0/108 [00:00<?, ?it/s]

Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.3696310566851395
            Iterations: 9
            Function evaluations: 9
            Gradient evaluations: 9
Optimization Complete
-----------------------------------


  1%|          | 1/108 [00:05<09:10,  5.15s/it]

Failed run: P=0.5, H=10.0 -> 'NoneType' object has no attribute 'text'
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.3696310566851395
            Iterations: 9
            Function evaluations: 9
            Gradient evaluations: 9
Optimization Complete
-----------------------------------


...

 99%|█████████▉| 107/108 [03:32<00:01,  1.92s/it]

Failed run: P=3.0, H=30.0 -> 'NoneType' object has no attribute 'text'
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.3696310566851395
            Iterations: 9
            Function evaluations: 9
            Gradient evaluations: 9
Optimization Complete
-----------------------------------


100%|██████████| 108/108 [03:34<00:00,  1.99s/it]

Failed run: P=3.0, H=30.0 -> 'NoneType' object has no attribute 'text'


""


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Ensure we have data
if not df_results.empty:
    
    # 1. 3D Scatter: Payload vs Hover vs Risk Adjusted Cost
    fig3d = px.scatter_3d(
        df_results,
        x='payload_kg',
        y='hover_min',
        z='risk_adj_cost_usd',
        color='lead_time_days',
        title='Design Space Trade-off: Payload vs Hover vs Cost (Color=Lead Time)',
        labels={'payload_kg': 'Payload (kg)', 'hover_min': 'Hover Time (min)', 'risk_adj_cost_usd': 'Risk Adj. Cost ($)'}
    )
    fig3d.show()

    # 2. Parallel Coordinates: Visualize parameter sensitivity
    fig_par = px.parallel_coordinates(
        df_results,
        color="risk_adj_cost_usd",
        labels={
            "payload_kg": "Payload (kg)",
            "hover_min": "Hover (min)",
            "risk_mult": "Risk Mult.",
            "logistics_days": "Logistics Buffer",
            "mtow_kg": "MTOW (kg)",
            "risk_adj_cost_usd": "Risk Adj. Cost ($)",
            "lead_time_days": "Lead Time (days)",
        },
        title="Parallel Coordinates Plot for Design Parameters & Unified Metrics"
    )
    fig_par.show()

    # 3. Scatter Matrix: Correlations between key variables
    fig_matrix = px.scatter_matrix(
        df_results,
        dimensions=["payload_kg", "hover_min", "mtow_kg", "total_cost_usd", "lead_time_days"],
        color="risk_adj_cost_usd",
        title="Scatter Matrix: Design & Supply Chain Metrics"
    )
    fig_matrix.show()
else:
    print("No DOE results to visualize.")